# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 09 · Complete RAG Pipeline

This notebook brings together all the components developed throughout the previous notebooks into the **complete RAG pipeline**. It provides a simple way to interact with the system and test the complete workflow end to end, from GitHub repository ingestion to the generation of the final answer.

To use the pipeline, the user only needs to specify the **URL of the GitHub repository** to analyse and select the desired **LLM execution mode**: a local model running through Ollama or an API-based model accessed through Groq.

Once the pipeline has been initialized, the only input that needs to be changed is the **user query** in the final section of the notebook. The query can then be modified and executed repeatedly to test different questions over the selected repository.

## Imports and Configuration

In [1]:
import time
from pathlib import Path
from IPython.display import Markdown, display
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
from github_rag.retrieval.retriever import Retriever
from github_rag.prompt_engineering.prompt_builder import build_prompt
from github_rag.LLM_generation.generator import LLMGenerator

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects that implements and evaluates a distributed Active Learning solution in PySpark.

In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

Configure the desired LLM according to your environment: locally via Ollama, or via API to use the Groq API with your API key.

In [3]:
# mode = "local"
mode = "API"

## 1. Pipeline Setup

Run the complete RAG pipeline, ensuring that all the components are properly initialized and ready to use.

In [4]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents after processing: {len(documents_processed)}")

# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents after chunking: {len(documents_chunking)}")

# Load the HuggingFace embedding model
start_time = time.time()
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = create_embedding_model(embedding_model_name)
print(f"Embedding model loaded time: {time.time() - start_time:.2f} seconds")

# Create the vector database storing
start_time = time.time()
vector_store = ChromaStore(
    mode="local",
    collection_name="github_repo"
)

# Create the database with documents and the embedding model
vector_store.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vector_store.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

# Get the available files for the Retriever component
available_files = vector_store.get_unique_file_paths()
print(f"Number of available files: {len(available_files)}")

# Create the Retriever object
retriever = Retriever(vector_store)

# Set the LLM model
LLM_generator = LLMGenerator(mode=mode)

Number of documents ingested: 12
Number of documents after processing: 45
Number of documents after chunking: 183
Embedding model loaded time: 8.98 seconds
Database generation time: 8.64 seconds
Documents provided: 183
Documents stored:   183
Number of available files: 12


## 2. Run the RAG System

The complete RAG pipeline is now ready to use. **Simply modify the `query` variable with your question and run the following cell.** You can execute the cell repeatedly with different queries to explore the repository.

In [17]:
query = "What is the clustering algorithm used in the diversity selection phase?"

In [18]:
# Run the RAG System
retrieval_chunks = retriever.retrieve(query, available_files)

# Build the prompt
prompt = build_prompt(retrieval_chunks, query)

# Query the LLM
response = LLM_generator.generate(prompt)
display(Markdown(response))

The diversity selection phase uses **Bisecting K‑Means** (the distributed implementation from Spark MLlib).  
This algorithm clusters the candidate instances and then picks the instance closest to each cluster centroid.  

Sources  
- [main.ipynb](https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb)  
- [scripts/AL_methods.py](https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py)